# 选修E1 · Day 1：Agent理论基础 · 上机练习（v5.0）

> **真实库**：LangChain + LangGraph + pydantic
> **核心范式**：ReAct（推理+行动）| Plan-Execute（规划+执行）
> **营销映射**：营销Agent的BDI认知结构 + 工具调用决策循环

本笔记本包含 **6个TODO填空**，完成后你将：
1. 用pydantic定义BDI Agent状态Schema
2. 用@tool装饰器定义营销工具
3. 用create_react_agent构建ReAct Agent
4. 运行Agent并分析Thought-Action-Observation轨迹
5. 用MemorySaver实现多轮对话记忆
6. 用StateGraph实现Plan-Execute模式

> 📦 真实库说明见 `data/README.md`
> 📖 理论讲义见 `notes.md`


In [ ]:
# === 导入真实库 ===
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, END
from pydantic import BaseModel, Field
from typing import Optional, TypedDict
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# === 真实营销数据（基于护肤品电商场景）===
PRODUCT_DB = {
    "透肌精华": "透肌焕亮精华液，299元，主打美白焕亮，含烟酰胺3%+维C衍生物，目标用户25-35岁都市白领。",
    "玻尿酸面霜": "玻尿酸保湿面霜，159元，主打深层补水，含双重玻尿酸，目标用户18-30岁女性。",
}
COMPETITOR_DB = {
    "雅诗兰黛": "雅诗兰黛小棕瓶精华，760元/30ml，市场占有率18%，优势：品牌力强、渠道完善；劣势：价格高、年轻化不足。",
    "兰蔻": "兰蔻小黑瓶精华，780元/30ml，市场占有率15%，优势：科技感强、专柜体验；劣势：下沉市场覆盖弱。",
}

# === 离线模拟LLM（无需API Key，预编排ReAct工具调用序列）===
class StubChatModel(BaseChatModel):
    """离线模拟LLM，预编排工具调用序列，保证无API Key可运行。
    替换为ChatOpenAI/ChatAnthropic即可使用真实LLM。"""
    responses: list = []
    call_index: int = 0

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        idx = self.call_index
        self.call_index += 1
        if idx < len(self.responses):
            resp = self.responses[idx]
        else:
            resp = AIMessage(content="任务完成。")
        return ChatResult(generations=[ChatGeneration(message=resp)])

    @property
    def _llm_type(self):
        return "stub"

    def bind_tools(self, tools, **kwargs):
        return self

print("真实库导入成功")
print(f"  产品库: {list(PRODUCT_DB.keys())}")
print(f"  竞品库: {list(COMPETITOR_DB.keys())}")
print(f"  StubChatModel: 离线模式（无API Key可运行）")


---
## TODO1：用pydantic定义BDI状态Schema

BDI（Belief-Desire-Intention）是经典Agent理论模型：
- **Belief（信念）**：Agent对世界的认知（产品信息、竞品信息、市场背景）
- **Desire（愿望）**：Agent想达成的目标（用户任务）
- **Intention（意图）**：Agent承诺执行的计划（步骤列表+当前步骤）

用pydantic的BaseModel将BDI形式化为Agent状态Schema，提供类型安全和自动校验。

> 参考教材 § Day 1 二、BDI架构

In [ ]:
# TODO1: 用pydantic定义BDI状态Schema
# 提示: 定义 Belief / Desire / Intention / BDIState 四个 BaseModel
# Belief: product_info, competitor_info, market_context
# Desire: task, success_criteria
# Intention: steps (list[str]), current_step (int)
# BDIState: belief, desire, intention

# TODO: 你的代码
raise NotImplementedError

# 验证BDI Schema（取消注释后测试）
# state = BDIState(
#     desire=Desire(task="为透肌精华制定营销策略"),
#     intention=Intention(steps=["搜索产品信息", "分析竞品", "撰写策略", "写入文件"])
# )
# print(f"Belief: {state.belief.model_dump()}")
# print(f"Desire: {state.desire.model_dump()}")
# print(f"Intention: {state.intention.model_dump()}")


---
## TODO2：用@tool装饰器定义营销工具

工具是Agent的"手"。在LangChain中，用`@tool`装饰器定义工具。
工具的**名称、docstring、参数类型**就是LLM看到的"接口契约"。

需要定义三个营销工具：
1. `search_product_info(product_name)` - 搜索产品信息（使用PRODUCT_DB）
2. `analyze_competitor(competitor_name)` - 分析竞品策略（使用COMPETITOR_DB）
3. `write_strategy(filename, content)` - 将策略写入文件

> 参考教材 § Day 1 四、工具使用

In [ ]:
# TODO2: 用@tool装饰器定义三个营销工具
# 提示: 使用 PRODUCT_DB 和 COMPETITOR_DB
# 工具1: search_product_info(product_name: str) -> str
# 工具2: analyze_competitor(competitor_name: str) -> str
# 工具3: write_strategy(filename: str, content: str) -> str
# 注意: docstring是LLM看到的接口契约，要写清楚！

# TODO: 你的代码
raise NotImplementedError

# 验证工具（取消注释后测试）
# tools = [search_product_info, analyze_competitor, write_strategy]
# print(f"定义了 {len(tools)} 个工具")
# result = search_product_info.invoke({"product_name": "透肌精华"})
# print(f"测试: {result}")


---
## TODO3：用create_react_agent构建ReAct Agent

ReAct（Reasoning + Acting）的核心循环：
```
Thought -> Action -> Observation -> Thought -> ... -> FINISH
```

用LangGraph的`create_react_agent`构建ReAct Agent：
- model: 使用StubChatModel（预编排工具调用序列）
- tools: 使用TODO2定义的三个工具
- prompt: 系统提示，定义Agent角色

> 参考教材 § Day 1 三、ReAct范式

In [ ]:
# TODO3: 用create_react_agent构建ReAct Agent
# 提示:
#   1. 创建 react_trajectory 列表，预编排4个AIMessage
#      - 前3个带 tool_calls (search_product_info, analyze_competitor, write_strategy)
#      - 最后一个带 content (最终回答)
#   2. 用 StubChatModel(responses=react_trajectory) 创建模型
#   3. 用 create_react_agent(model, tools, prompt=...) 构建Agent

# TODO: 你的代码
raise NotImplementedError

# 验证Agent（取消注释后测试）
# print(f"ReAct Agent构建成功，工具: {[t.name for t in tools]}")


---
## TODO4：运行Agent并分析ReAct轨迹

运行Agent处理营销任务，观察Thought-Action-Observation循环：
- 统计工具调用次数
- 统计模型调用次数
- 分析Agent的工具选择顺序是否符合天道推演的因果预期

> 天道推演视角：每个Action改变Belief，影响下一次Thought，形成因果链

In [ ]:
# TODO4: 运行Agent处理营销任务，观察ReAct轨迹
# 提示:
#   1. 用 agent.invoke({"messages": [("user", task)]}) 运行Agent
#   2. 遍历 result["messages"]，打印每个消息的类型和内容
#   3. 统计 tool_calls_count（工具调用次数）和 thought_count（AI有内容的消息数）
#   4. 打印轨迹分析总结

task = "为透肌精华制定营销策略，竞品分析雅诗兰黛，并写入策略文件"

# TODO: 你的代码
raise NotImplementedError


---
## TODO5：用MemorySaver实现多轮对话记忆

MemorySaver是LangGraph的checkpointer，按`thread_id`隔离不同会话。
同一thread_id的多轮对话共享上下文历史。

实现：
1. 构建带MemorySaver的Agent
2. 第一轮对话：查询产品信息
3. 第二轮对话：Agent应记住第一轮的内容

> 参考教材 § Day 1 关键回顾4：Agent记忆

In [ ]:
# TODO5: 用MemorySaver添加短期记忆，实现多轮对话
# 提示:
#   1. 创建新的 StubChatModel (3个预编排响应)
#   2. 用 MemorySaver() 创建 checkpointer
#   3. 用 create_react_agent(model, tools, prompt=..., checkpointer=memory) 构建Agent
#   4. 用 config = {"configurable": {"thread_id": "session-1"}} 隔离会话
#   5. 第一轮: 查询"玻尿酸面霜"信息
#   6. 第二轮: 问"你刚才查的是什么产品？"
#   7. 验证第2轮消息数 > 第1轮（记忆生效）

# TODO: 你的代码
raise NotImplementedError


---
## TODO6：用StateGraph实现Plan-Execute模式

Plan-Execute与ReAct的核心区别：
- **ReAct**：边推理边执行，每步可根据观测调整下一步
- **Plan-Execute**：先一次性规划所有步骤，再顺序执行

用LangGraph的StateGraph实现：
1. `plan_node`：生成完整计划
2. `execute_node`：顺序执行每个步骤
3. 条件边：判断是否还有未执行步骤

> 参考教材 § Day 1 三、ReAct的局限性与改进

In [ ]:
# TODO6: 用StateGraph实现Plan-Execute模式
# 提示:
#   1. 定义 PlanExecuteState (TypedDict): task, plan, current_step, results, final_answer
#   2. plan_node: 生成4步计划（搜索产品/分析竞品/撰写策略/写入文件）
#   3. execute_node: 根据步骤描述执行，使用 PRODUCT_DB / COMPETITOR_DB
#   4. should_continue: current_step < len(plan) 则 continue，否则 end
#   5. 构建图: plan -> execute -> (continue: execute | end: END)
#   6. 运行并打印计划和执行结果
#   7. 对比 ReAct vs Plan-Execute 的差异

# TODO: 你的代码
raise NotImplementedError
